# 🚀 Qwen Model Eğitimi - Google Colab Başlatıcı

## 🏁 Hızlı Başlangıç

**Ngrok veya karmaşık ayarlara ihtiyacınız YOKTUR.** 
Modeli eğitmek için sadece aşağıdaki butona tıklayın ve hücreleri sırasıyla çalıştırın.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cebrailbagatarhan/yapay-zeka-sistemi/blob/main/colab_runner.ipynb)

### 📋 Adımlar:
1. Yukarıdaki **"Open in Colab"** butonuna tıklayın.
2. Açılan sayfada **Runtime > Change runtime type** menüsünden **T4 GPU** seçin.
3. Yukarıdan aşağıya doğru hücreleri sırasıyla çalıştırın (Play butonu veya CTRL+ENTER).
4. **Bölüm 6 (Model Eğitimi)** kısmına geldiğinizde eğitim başlayacaktır.

---
**Not:** Bu notebook GitHub'dan otomatik olarak en güncel kodları çekecektir.


# 🚀 Qwen Model Eğitimi - Google Colab Optimized

Bu notebook, Google Colab'ın T4/A100 GPU'ları için optimize edilmiştir.

**⚡ Özellikler:**
- 4-bit Quantization (QLoRA)
- Flash Attention 2.0
- Gradient Checkpointing
- Mixed Precision (FP16/BF16)
- Paged AdamW 8-bit Optimizer

## 1️⃣ GPU Kontrolü ve Kurulum

In [4]:
# 🔍 GPU Kontrolü ve Colab Ortam Optimizasyonu
import torch
import os

print("🔍 COLAB GPU KONTROLÜ")
print("=" * 60)

# Colab ortamını kontrol et
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False
print(f"📍 Ortam: {'Google Colab' if IN_COLAB else 'Lokal'}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    # GPU tipini belirle
    if 'A100' in gpu_name:
        GPU_TYPE = 'A100'
        BATCH_SIZE = 4
        MAX_SEQ_LEN = 1024
        USE_BF16 = True
    elif 'V100' in gpu_name:
        GPU_TYPE = 'V100'
        BATCH_SIZE = 2
        MAX_SEQ_LEN = 768
        USE_BF16 = False
    elif 'T4' in gpu_name:
        GPU_TYPE = 'T4'
        BATCH_SIZE = 2
        MAX_SEQ_LEN = 512
        USE_BF16 = False
    else:
        GPU_TYPE = 'OTHER'
        BATCH_SIZE = 1
        MAX_SEQ_LEN = 512
        USE_BF16 = False
    
    print(f"✅ GPU Bulundu: {gpu_name}")
    print(f"💾 GPU Bellek: {gpu_memory:.1f} GB")
    print(f"🔥 CUDA Version: {torch.version.cuda}")
    print(f"⚡ PyTorch Version: {torch.__version__}")
    print(f"\n🎯 Optimize Edilmiş Ayarlar ({GPU_TYPE}):")
    print(f"   • Batch Size: {BATCH_SIZE}")
    print(f"   • Max Sequence Length: {MAX_SEQ_LEN}")
    print(f"   • BF16: {'Aktif' if USE_BF16 else 'FP16 kullanılacak'}")
else:
    print("❌ GPU bulunamadı!")
    print("💡 Runtime > Change runtime type > T4 GPU seçin")
    GPU_TYPE = 'CPU'
    BATCH_SIZE = 1
    MAX_SEQ_LEN = 256
    USE_BF16 = False

# CUDA optimizasyonları
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    print("\n✅ CUDA TF32 & cuDNN optimizasyonları aktif!")

🔍 COLAB GPU KONTROLÜ
📍 Ortam: Google Colab
❌ GPU bulunamadı!
💡 Runtime > Change runtime type > T4 GPU seçin


In [ ]:
# 📦 Gerekli Kütüphaneleri Yükle (Colab Optimized)
print("📦 KÜTÜPHANELER YÜKLENİYOR...")
print("=" * 60)

# Temel kütüphaneler
!pip install -q transformers>=4.36.0 accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0

# Fine-tuning kütüphaneleri
!pip install -q peft>=0.7.0 trl>=0.7.0 datasets

# Flash Attention (A100/H100 için)
try:
    import subprocess
    result = subprocess.run(['pip', 'show', 'flash-attn'], capture_output=True, text=True)
    if 'not found' in result.stderr.lower() or result.returncode != 0:
        if GPU_TYPE in ['A100', 'H100']:
            print("⚡ Flash Attention yükleniyor (A100 için)...")
            !pip install -q flash-attn --no-build-isolation
except:
    pass

# Web araştırma kütüphaneleri
!pip install -q httpx beautifulsoup4 duckduckgo-search wikipedia
!pip install -q sentencepiece protobuf scipy

# Bellek temizle
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Tüm kütüphaneler yüklendi!")
print("   • Transformers (LLM)")
print("   • PEFT + TRL (Fine-tuning)")
print("   • BitsAndBytes (Quantization)")
print("   • Accelerate (GPU Opt.)")

📦 KÜTÜPHANELER YÜKLENİYOR...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 24.6 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 82.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.4 MB/s eta 0:00:00

✅ Tüm kütüphaneler yüklendi!


In [4]:
# 📥 GitHub'dan Projeyi İndir
import os

print("📥 PROJE İNDİRİLİYOR...")
print("=" * 50)

# Repo'yu klonla
if not os.path.exists('yapay-zeka-sistemi'):
    !git clone https://github.com/cebrailbagatarhan/yapay-zeka-sistemi.git
    print("✅ Proje indirildi!")
else:
    print("✅ Proje zaten mevcut, güncelleniyor...")
    %cd yapay-zeka-sistemi
    !git pull
    %cd ..

# Proje dizinine geç
%cd yapay-zeka-sistemi
print(f"\n📂 Çalışma dizini: {os.getcwd()}")

📥 PROJE İNDİRİLİYOR...
Cloning into 'yapay-zeka-sistemi'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ Proje indirildi!
[Errno 2] No such file or directory: 'yapay-zeka-sistemi'
/content

📂 Çalışma dizini: /content


## 2️⃣ Qwen Modelini İndir ve Yükle

In [ ]:
# 🤖 Qwen Modelini Colab-Optimized Yükle
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print("🤖 QWEN 2.5-1.5B MODEL YÜKLENİYOR (Colab Optimized)...")
print("=" * 60)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# GPU tipine göre quantization config
if GPU_TYPE == 'A100':
    # A100: 4-bit with BF16 compute
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    attn_implementation = "flash_attention_2"
    print("⚡ A100 Modu: Flash Attention 2 + BF16")
elif GPU_TYPE == 'T4':
    # T4: 4-bit with FP16 compute
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    attn_implementation = "eager"
    print("⚡ T4 Modu: 4-bit Quantization + FP16")
else:
    # Diğer GPU'lar
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    attn_implementation = "eager"
    print("⚡ Standart Mod: 4-bit Quantization")

print("\n📥 Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True,
    padding_side="right"
)
tokenizer.pad_token = tokenizer.eos_token

print("📥 Model yükleniyor...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation=attn_implementation if GPU_TYPE == 'A100' else None,
        torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    )
except Exception as e:
    print(f"⚠️ Flash Attention kullanılamadı, standart mod: {e}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )

# Bellek bilgisi
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\n✅ Model yüklendi!")
    print(f"💾 GPU Kullanımı: {allocated:.2f} GB (ayrılmış: {reserved:.2f} GB)")
    print(f"🚀 4-bit quantization ile ~%75 bellek tasarrufu!")

🤖 QWEN 2.5-1.5B MODEL YÜKLENİYOR...
📥 Tokenizer yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

📥 Model yükleniyor (4-bit quantized)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:
# 🧪 Model Testi
def ask_qwen(prompt, max_tokens=256):
    """Qwen'e soru sor"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir asistansın. Türkçe ve İngilizce konuşabilirsin."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    
    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return response

# Test
print("🧪 MODEL TESTİ")
print("=" * 50)

test_prompts = [
    "Merhaba! Nasılsın?",
    "Python'da liste nasıl oluşturulur?",
    "5 + 3 * 2 kaç eder?"
]

for prompt in test_prompts:
    print(f"\n👤 Soru: {prompt}")
    response = ask_qwen(prompt)
    print(f"🤖 Qwen: {response}")
    print("-" * 50)

🧪 MODEL TESTİ

👤 Soru: Merhaba! Nasılsın?


KeyboardInterrupt: 

## 3️⃣ Derin Web Araştırma Sistemi

In [7]:
# 🔍 Derin Web Araştırma Sistemi
import sys
sys.path.append('/content/yapay-zeka-sistemi')

from src.deep_web_researcher import DeepWebResearcher

print("🔍 DERİN WEB ARAŞTIRMA SİSTEMİ")
print("=" * 50)

# Araştırmacıyı başlat
researcher = DeepWebResearcher()
print("✅ Web araştırma sistemi hazır!")

# Test araştırması
def deep_research_with_qwen(topic, max_sources=5):
    """Derin araştırma + Qwen analizi"""
    print(f"\n🔍 ARAŞTIRMA: {topic}")
    print("=" * 50)
    
    # Web araştırması
    print("📡 Web taraması yapılıyor (async)...")
    results = researcher.deep_research(topic, max_sources=max_sources, use_async=True)
    
    if not results:
        print("❌ Sonuç bulunamadı")
        return
    
    print(f"\n✅ {len(results)} kaynak bulundu!\n")
    
    # Kaynakları göster
    print("📚 KAYNAKLAR:")
    for i, r in enumerate(results[:3], 1):
        print(f"{i}. {r['title'][:60]}...")
        print(f"   🔗 {r['url'][:50]}...")
    
    # Qwen ile özet
    print("\n🤖 Qwen analiz ediyor...")
    
    combined_info = f"'{topic}' hakkında bilgiler:\n\n"
    for r in results[:5]:
        combined_info += f"- {r['title']}: {r.get('snippet', '')[:200]}\n"
    
    prompt = f"{combined_info}\n\nYukarıdaki bilgilere dayanarak '{topic}' hakkında kısa bir özet yaz:"
    
    summary = ask_qwen(prompt, max_tokens=300)
    
    print("\n" + "=" * 50)
    print("📊 QWEN ÖZETİ:")
    print("=" * 50)
    print(summary)
    
    return summary

ModuleNotFoundError: No module named 'src'

In [8]:
# 🔬 Araştırma Testi
deep_research_with_qwen("Machine Learning nedir", max_sources=5)

NameError: name 'deep_research_with_qwen' is not defined

## 4️⃣ İnteraktif Sohbet Modu

In [ ]:
# 💬 İnteraktif Sohbet
from IPython.display import clear_output

print("💬 İNTERAKTİF SOHBET MODU")
print("=" * 50)
print("\n📋 Komutlar:")
print("  • Direkt soru yazın - Qwen cevaplar")
print("  • 'araştır: [konu]' - Web araştırması yapar")
print("  • 'q' - Çıkış")
print("\n" + "=" * 50)

conversation_history = []

while True:
    user_input = input("\n👤 Siz: ").strip()
    
    if user_input.lower() in ['q', 'quit', 'exit', 'çıkış']:
        print("\n👋 Görüşmek üzere!")
        break
    
    if not user_input:
        continue
    
    if user_input.lower().startswith("araştır:"):
        topic = user_input[8:].strip()
        if topic:
            deep_research_with_qwen(topic, max_sources=5)
    else:
        # Direkt soru
        response = ask_qwen(user_input)
        print(f"\n🤖 Qwen: {response}")
        
        # Geçmişe ekle
        conversation_history.append({"user": user_input, "assistant": response})

💬 İNTERAKTİF SOHBET MODU

📋 Komutlar:
  • Direkt soru yazın - Qwen cevaplar
  • 'araştır: [konu]' - Web araştırması yapar
  • 'q' - Çıkış



## 5️⃣ Bonus: Gradio Arayüzü

In [ ]:
# 🎨 Gradio Web Arayüzü
!pip install -q gradio

import gradio as gr

def chat_interface(message, history):
    """Gradio chat interface"""
    if message.lower().startswith("araştır:"):
        topic = message[8:].strip()
        results = researcher.deep_research(topic, max_sources=3, use_async=True)
        
        if results:
            info = f"'{topic}' hakkında {len(results)} kaynak bulundu:\n\n"
            for r in results[:3]:
                info += f"• {r['title']}: {r.get('snippet', '')[:100]}...\n"
            
            prompt = f"{info}\n\nBu bilgilere dayanarak özet yap:"
            response = ask_qwen(prompt, max_tokens=300)
        else:
            response = "Araştırma sonucu bulunamadı."
    else:
        response = ask_qwen(message)
    
    return response

# Gradio arayüzü
demo = gr.ChatInterface(
    fn=chat_interface,
    title="🤖 Yapay Zeka Sistemi",
    description="Qwen 2.5-1.5B + Derin Web Araştırma\n\n💡 'araştır: [konu]' yazarak web araştırması yapabilirsiniz.",
    examples=[
        "Merhaba! Nasılsın?",
        "Python'da döngü nasıl yazılır?",
        "araştır: Yapay zeka nedir"
    ],
    theme="soft"
)

print("🎨 Gradio arayüzü başlatılıyor...")
demo.launch(share=True, debug=False)

## 6️⃣ Model Eğitimi (Fine-Tuning)

Aşağıdaki hücrelerde Qwen modelini özel veri setlerimizle eğiteceğiz:
- **Conversational Dataset**: Sohbet ve yardım konuşmaları
- **Reasoning Dataset**: Teknik açıklamalar ve Chain-of-Thought
- **CoT Dataset**: Matematik ve mantık problemleri

In [1]:
# 📦 Fine-Tuning için Gerekli Kütüphaneleri Yükle
print("📦 FINE-TUNING KÜTÜPHANELERİ YÜKLENİYOR...")
print("=" * 50)

!pip install -q peft trl datasets accelerate
!pip install -q bitsandbytes>=0.41.0
!pip install -q scipy

print("\n✅ Fine-tuning kütüphaneleri yüklendi!")
print("   • PEFT (LoRA için)")
print("   • TRL (Trainer)")
print("   • Datasets (Veri yönetimi)")
print("   • Accelerate (GPU optimizasyonu)")

📦 FINE-TUNING KÜTÜPHANELERİ YÜKLENİYOR...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_p

In [ ]:
# 📚 Veri Setlerini Yükle ve Hazırla
import json
import os
from datasets import Dataset

print("📚 VERİ SETLERİ HAZIRLANIYOR...")
print("=" * 50)

# Veri setlerini yükle
def load_json_data(filepath):
    """JSON dosyasını yükle"""
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

# Proje dizininde mi kontrol et
base_path = '/content/yapay-zeka-sistemi' if os.path.exists('/content/yapay-zeka-sistemi') else '.'

# Veri setlerini yükle
conversational_data = load_json_data(f'{base_path}/data/training/conversational_dataset.json')
reasoning_data = load_json_data(f'{base_path}/data/training/reasoning_chat_dataset.json')
cot_data = load_json_data(f'{base_path}/data/examples/cot_dataset.json')

print(f"✅ Conversational Dataset: {len(conversational_data)} örnek")
print(f"✅ Reasoning Dataset: {len(reasoning_data)} örnek")
print(f"✅ CoT Dataset: {len(cot_data)} örnek")

# Tüm verileri birleştir ve formatla
all_training_data = []

# Conversational data formatla
for item in conversational_data:
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': item.get('response', ''),
        'type': 'conversational'
    })

# Reasoning data formatla
for item in reasoning_data:
    reasoning_text = item.get('reasoning', '')
    answer_text = item.get('answer', '')
    full_response = f"Düşünce Süreci:\n{reasoning_text}\n\nSonuç: {answer_text}"
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': full_response,
        'type': 'reasoning'
    })

# CoT data formatla
for item in cot_data:
    reasoning_text = item.get('reasoning', '')
    answer_text = item.get('answer', '')
    full_response = f"Adım adım çözüm:\n{reasoning_text}\n\nCevap: {answer_text}"
    all_training_data.append({
        'instruction': item.get('question', ''),
        'input': '',
        'output': full_response,
        'type': 'cot'
    })

print(f"\n🎯 Toplam Eğitim Verisi: {len(all_training_data)} örnek")

# Dataset oluştur
train_dataset = Dataset.from_list(all_training_data)
print(f"✅ Dataset oluşturuldu: {train_dataset}")

# Örnek göster
print("\n📝 Örnek Veri:")
print("-" * 50)
sample = all_training_data[0]
print(f"Soru: {sample['instruction'][:100]}...")
print(f"Cevap: {sample['output'][:150]}...")

In [ ]:
# 🔧 LoRA Konfigürasyonu (Colab GPU Optimized)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("🔧 LoRA KONFİGÜRASYONU (Colab Optimized)...")
print("=" * 60)

# GPU tipine göre LoRA rank ayarla
if GPU_TYPE == 'A100':
    LORA_R = 32       # A100: Daha yüksek rank
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.05
elif GPU_TYPE == 'T4':
    LORA_R = 16       # T4: Orta rank
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.1
else:
    LORA_R = 8        # Diğer: Düşük rank
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.1

# LoRA parametreleri
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    # Colab optimizasyonları
    modules_to_save=None,
)

print(f"📊 LoRA Parametreleri ({GPU_TYPE} için):")
print(f"   • Rank (r): {LORA_R}")
print(f"   • Alpha: {LORA_ALPHA}")
print(f"   • Dropout: {LORA_DROPOUT}")
print(f"   • Target: Attention + MLP katmanları")

# Modeli LoRA için hazırla
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model = get_peft_model(model, lora_config)

# Gradient checkpointing aktif et
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

# Eğitilebilir parametre sayısını göster
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
percentage = 100 * trainable_params / total_params

print(f"\n✅ LoRA Model Hazır!")
print(f"📊 Eğitilebilir: {trainable_params:,} / {total_params:,} ({percentage:.2f}%)")
print(f"🚀 Bellek tasarrufu: ~{100-percentage:.1f}%!")

# Bellek durumu
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 GPU Bellek: {allocated:.2f} GB")

In [ ]:
# 📝 Veriyi Chat Formatına Dönüştür
def format_instruction(sample):
    """Qwen chat formatına dönüştür"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir Türkçe yapay zeka asistanısın. Adım adım düşünerek açık ve detaylı cevaplar verirsin."},
        {"role": "user", "content": sample['instruction']},
        {"role": "assistant", "content": sample['output']}
    ]
    
    # Chat template uygula
    formatted = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

print("📝 VERİ FORMATLANIYOR...")
print("=" * 50)

# Dataset'i formatla
formatted_dataset = train_dataset.map(format_instruction)

print(f"✅ {len(formatted_dataset)} örnek formatlandı!")

# Örnek göster
print("\n📄 Formatlanmış Örnek:")
print("-" * 50)
print(formatted_dataset[0]['text'][:500] + "...")

In [ ]:
# 🚀 Eğitim Ayarları (Colab GPU Optimized)
from transformers import TrainingArguments
from trl import SFTTrainer

print("🚀 EĞİTİM AYARLARI (Colab Optimized)...")
print("=" * 60)

# GPU tipine göre eğitim parametreleri
if GPU_TYPE == 'A100':
    EPOCHS = 3
    GRAD_ACCUM = 2
    LR = 2e-4
    WARMUP = 0.05
    print("⚡ A100 Modu: Yüksek performans ayarları")
elif GPU_TYPE == 'T4':
    EPOCHS = 3
    GRAD_ACCUM = 4
    LR = 2e-4
    WARMUP = 0.1
    print("⚡ T4 Modu: Dengeli performans ayarları")
else:
    EPOCHS = 2
    GRAD_ACCUM = 8
    LR = 1e-4
    WARMUP = 0.1
    print("⚡ Standart Mod: Güvenli ayarlar")

# Eğitim parametreleri
training_args = TrainingArguments(
    output_dir="./qwen-turkish-finetuned",
    
    # Epoch ve batch ayarları
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Öğrenme oranı
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=WARMUP,
    lr_scheduler_type="cosine",
    
    # Optimizasyon
    optim="paged_adamw_8bit",
    fp16=not USE_BF16,
    bf16=USE_BF16,
    
    # Gradient
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    
    # Logging ve kayıt
    logging_steps=5,
    logging_first_step=True,
    save_steps=50,
    save_total_limit=2,
    save_strategy="steps",
    
    # Performans
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    group_by_length=True,
    
    # Diğer
    report_to="none",
    remove_unused_columns=False,
    push_to_hub=False,
)

# SFT Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=True if len(formatted_dataset) > 100 else False,  # Büyük dataset için packing
)

effective_batch = BATCH_SIZE * GRAD_ACCUM
total_steps = (len(formatted_dataset) // effective_batch) * EPOCHS

print(f"\n✅ Trainer hazır!")
print(f"\n📊 Eğitim Parametreleri ({GPU_TYPE}):")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch Size: {BATCH_SIZE} (effective: {effective_batch})")
print(f"   • Learning Rate: {LR}")
print(f"   • Max Sequence: {MAX_SEQ_LEN}")
print(f"   • Tahmini Adım: ~{total_steps}")
print(f"   • Precision: {'BF16' if USE_BF16 else 'FP16'}")
print(f"   • Optimizer: Paged AdamW 8-bit")
print(f"   • Gradient Checkpointing: ✅")

In [ ]:
# 🎯 EĞİTİMİ BAŞLAT! (Colab Optimized)
import time
import gc

print("🎯 MODEL EĞİTİMİ BAŞLIYOR!")
print("=" * 60)
print(f"⚡ GPU: {GPU_TYPE}")
print(f"📊 Dataset: {len(formatted_dataset)} örnek")
print(f"⏳ Tahmini süre: {GPU_TYPE == 'T4' and '5-10' or '2-5'} dakika\n")

# Bellek temizliği
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_mem = torch.cuda.memory_allocated() / 1024**3
    print(f"💾 Başlangıç GPU Bellek: {start_mem:.2f} GB")

print("\n" + "=" * 60)
print("🔄 Eğitim başlıyor... Lütfen bekleyin.")
print("=" * 60 + "\n")

start_time = time.time()

try:
    # Eğitimi başlat
    train_result = trainer.train()
    
    end_time = time.time()
    training_time = end_time - start_time
    
    print("\n" + "=" * 60)
    print("✅ EĞİTİM BAŞARIYLA TAMAMLANDI!")
    print("=" * 60)
    print(f"⏱️ Toplam Süre: {training_time/60:.1f} dakika")
    print(f"📊 Final Loss: {train_result.training_loss:.4f}")
    
    if torch.cuda.is_available():
        end_mem = torch.cuda.memory_allocated() / 1024**3
        peak_mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"💾 Final GPU Bellek: {end_mem:.2f} GB")
        print(f"📈 Peak GPU Bellek: {peak_mem:.2f} GB")
        
except Exception as e:
    print(f"\n❌ Eğitim hatası: {e}")
    print("\n💡 Çözüm önerileri:")
    print("   1. Runtime'ı yeniden başlatın")
    print("   2. Batch size'ı düşürün")
    print("   3. Max sequence length'i azaltın")
    raise e

In [ ]:
# 💾 Modeli Kaydet (Colab + Google Drive)
import os
import shutil

print("💾 MODEL KAYDEDİLİYOR...")
print("=" * 60)

# Lokal kaydet
LOCAL_PATH = "./qwen-turkish-finetuned"
model.save_pretrained(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)
print(f"✅ Lokal kaydedildi: {LOCAL_PATH}")

# Google Drive'a kaydet
try:
    from google.colab import drive
    
    # Drive'ı mount et
    if not os.path.exists('/content/drive'):
        print("\n📁 Google Drive bağlanıyor...")
        drive.mount('/content/drive')
    
    # Hedef klasör
    DRIVE_PATH = '/content/drive/MyDrive/AI-Models/qwen-turkish-finetuned'
    
    # Klasörü oluştur
    os.makedirs(os.path.dirname(DRIVE_PATH), exist_ok=True)
    
    # Varsa sil
    if os.path.exists(DRIVE_PATH):
        shutil.rmtree(DRIVE_PATH)
    
    # Kopyala
    shutil.copytree(LOCAL_PATH, DRIVE_PATH)
    
    print(f"✅ Google Drive'a kaydedildi!")
    print(f"   📂 {DRIVE_PATH}")
    
    # Dosya boyutlarını göster
    total_size = 0
    for root, dirs, files in os.walk(DRIVE_PATH):
        for f in files:
            fp = os.path.join(root, f)
            total_size += os.path.getsize(fp)
    print(f"   📊 Toplam boyut: {total_size/1024/1024:.1f} MB")
    
except ImportError:
    print("ℹ️ Google Drive mevcut değil (lokal ortam)")
except Exception as e:
    print(f"⚠️ Drive kayıt hatası: {e}")
    print("   Lokal kayıt başarılı, Drive'a manuel kopyalayabilirsiniz.")

# Bellek temizliği
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    
print("\n🎉 Model başarıyla kaydedildi!")

In [ ]:
# 🧪 Eğitilmiş Modeli Test Et
print("🧪 EĞİTİLMİŞ MODEL TESTİ")
print("=" * 50)

def ask_finetuned_model(prompt, max_tokens=256):
    """Fine-tuned modele soru sor"""
    messages = [
        {"role": "system", "content": "Sen yardımcı bir Türkçe yapay zeka asistanısın. Adım adım düşünerek açık ve detaylı cevaplar verirsin."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1
        )
    
    response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
    return response

# Test soruları
test_questions = [
    "Merhaba! Bana Python'da döngüler hakkında bilgi verir misin?",
    "5 + 3 × 2 kaç eder? Adım adım açıkla.",
    "Machine learning nedir? Basitçe açıklar mısın?",
    "API nedir ve neden kullanılır?"
]

print("\n📊 EĞİTİM ÖNCESİ vs SONRASI KARŞILAŞTIRMASI\n")

for i, question in enumerate(test_questions, 1):
    print(f"{'='*60}")
    print(f"📌 SORU {i}: {question}")
    print(f"{'='*60}")
    
    response = ask_finetuned_model(question)
    print(f"\n🤖 Fine-tuned Model Cevabı:")
    print(f"{response}")
    print()

## 7️⃣ Eğitim Metrikleri ve Analiz

In [ ]:
# 📊 Eğitim Grafiklerini Çiz
import matplotlib.pyplot as plt

print("📊 EĞİTİM METRİKLERİ")
print("=" * 50)

# Eğitim loglarını al
training_logs = trainer.state.log_history

# Loss değerlerini çıkar
train_losses = []
steps = []

for log in training_logs:
    if 'loss' in log:
        train_losses.append(log['loss'])
        steps.append(log.get('step', len(steps)))

# Grafik çiz
if train_losses:
    plt.figure(figsize=(12, 4))
    
    # Loss grafiği
    plt.subplot(1, 2, 1)
    plt.plot(steps, train_losses, 'b-', linewidth=2, label='Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('🎯 Training Loss Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Loss dağılımı
    plt.subplot(1, 2, 2)
    plt.hist(train_losses, bins=20, color='steelblue', edgecolor='white')
    plt.xlabel('Loss Value')
    plt.ylabel('Frequency')
    plt.title('📈 Loss Distribution')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_metrics.png', dpi=150)
    plt.show()
    
    # İstatistikler
    print(f"\n📊 Eğitim İstatistikleri:")
    print(f"   • Başlangıç Loss: {train_losses[0]:.4f}")
    print(f"   • Final Loss: {train_losses[-1]:.4f}")
    print(f"   • İyileşme: {((train_losses[0] - train_losses[-1]) / train_losses[0] * 100):.1f}%")
    print(f"   • Min Loss: {min(train_losses):.4f}")
    print(f"   • Ortalama Loss: {sum(train_losses)/len(train_losses):.4f}")
else:
    print("⚠️ Eğitim logları bulunamadı")

In [ ]:
# 🎮 İnteraktif Fine-tuned Model Sohbeti
print("🎮 FINE-TUNED MODEL İLE SOHBET")
print("=" * 50)
print("\n💡 Artık eğitilmiş modelinizle sohbet edebilirsiniz!")
print("📋 Komutlar:")
print("   • Direkt soru yazın")
print("   • 'q' ile çıkış")
print("\n" + "=" * 50)

while True:
    user_input = input("\n👤 Siz: ").strip()
    
    if user_input.lower() in ['q', 'quit', 'exit', 'çıkış']:
        print("\n👋 İyi çalışmalar!")
        break
    
    if not user_input:
        continue
    
    response = ask_finetuned_model(user_input)
    print(f"\n🤖 Model: {response}")

---

## 📊 Colab GPU Performans Bilgileri

| GPU | Batch | Seq Len | Precision | Tahmini Süre |
|-----|-------|---------|-----------|--------------|
| **A100** | 4 | 1024 | BF16 | 2-5 dk |
| **V100** | 2 | 768 | FP16 | 5-10 dk |
| **T4** | 2 | 512 | FP16 | 5-10 dk |
| **CPU** | 1 | 256 | FP32 | 30+ dk |

## ⚡ Optimizasyonlar

| Özellik | Açıklama |
|---------|----------|
| **QLoRA** | 4-bit Quantization + LoRA |
| **Flash Attention** | A100 için hızlandırma |
| **Gradient Checkpointing** | %50 bellek tasarrufu |
| **Paged AdamW 8-bit** | Hafif optimizer |
| **TF32 / cuDNN** | Matrix çarpımı optimizasyonu |

## 💡 İpuçları

1. **GPU seçin**: `Runtime > Change runtime type > T4 GPU`
2. **Bellek doluyorsa**: Runtime'ı yeniden başlatın
3. **Hata alırsanız**: Batch size veya seq length azaltın
4. **Modeli kaybedin**: Google Drive'a kaydedin!

---

🎉 **Colab'da eğitim tamamlandı!**